In [1]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [2]:
# loading the data set 
df = pd.read_csv("../data/twitter_training.csv")
df

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


### preprocessing 

In [3]:
# column cleaning 
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='str')

In [4]:
# drop the 2401 columns it irrelevant 
df.drop(columns=["2401"], inplace=True)
df

,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,Borderlands,Positive,I am coming to the borders and I will kill you...
1,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,Borderlands,Positive,im coming on borderlands and i will murder you...
3,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...
74676,Nvidia,Positive,Just realized that the Windows partition of my...
74677,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,Nvidia,Positive,Just realized between the windows partition of...


In [5]:
# rename columns 
df.rename(columns={
    'Borderlands':'game_name',
    'Positive':'sentiment',
    'im getting on borderlands and i will murder you all ,':'tweet'

},inplace=True)

df.columns

Index(['game_name', 'sentiment', 'tweet'], dtype='str')

In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   game_name  74681 non-null  str  
 1   sentiment  74681 non-null  str  
 2   tweet      73995 non-null  str  
dtypes: str(3)
memory usage: 1.7 MB


In [7]:
# drop null values 
df.isna().sum()
df.dropna(inplace=True)
df.isna().sum()

game_name    0
sentiment    0
tweet        0
dtype: int64

In [8]:
# check for duplicates 
df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [9]:
df.shape

(70957, 3)

In [10]:
# ensure balance between sentiment by reducing trimming data
df["sentiment"].value_counts()

sentiment
Negative      21565
Positive      19548
Neutral       17398
Irrelevant    12446
Name: count, dtype: int64

In [11]:
df_negative = df[df.sentiment == 'Negative'].iloc[:2000]
df_positive = df[df.sentiment == 'Positive'].iloc[:2000]
df_neutral = df[df.sentiment == 'Neutral'].iloc[:2000]
df_irrelevant = df[df.sentiment == 'Irrelevant'].iloc[:2000]

In [12]:
# now our df has 8000 dataset 
df = pd.concat([df_negative,df_positive,df_positive,df_irrelevant],axis=0)
df.reset_index(drop=True,inplace=True)
df

,game_name,sentiment,tweet
0,Borderlands,Negative,the biggest dissappoinment in my life came out...
1,Borderlands,Negative,The biggest disappointment of my life came a y...
2,Borderlands,Negative,the biggest dissappoinment in my life coming o...
3,Borderlands,Negative,For the biggest male dissappoinment in my life...
4,Borderlands,Negative,the biggest dissappoinment in my life came bac...
...,...,...,...
7995,Xbox(Xseries),Irrelevant,Latest news about Xbox Geek! paper.li / XboxGe...
7996,Xbox(Xseries),Irrelevant,Latest xbox geek! paper.li / XboxGeekNews / 1....
7997,Xbox(Xseries),Irrelevant,The latest xbox hack blog! paper.li/XboxGeekNe...
7998,Xbox(Xseries),Irrelevant,The latest xbox from geek news! smart paper. l...


In [13]:
df["game_name"].value_counts()

game_name
CallOfDutyBlackopsColdWar    2712
Borderlands                  2573
Amazon                       1194
Overwatch                    1123
Xbox(Xseries)                 398
Name: count, dtype: int64

In [14]:
# categorical to numeric 
df_game = pd.get_dummies(df['game_name']).astype("int")
df_game

,Amazon,Borderlands,CallOfDutyBlackopsColdWar,Overwatch,Xbox(Xseries)
0,0,1,0,0,0
1,0,1,0,0,0
2,0,1,0,0,0
3,0,1,0,0,0
4,0,1,0,0,0
...,...,...,...,...,...
7995,0,0,0,0,1
7996,0,0,0,0,1
7997,0,0,0,0,1
7998,0,0,0,0,1


In [15]:
df.drop(columns=["game_name"],inplace=True)
df

,sentiment,tweet
0,Negative,the biggest dissappoinment in my life came out...
1,Negative,The biggest disappointment of my life came a y...
2,Negative,the biggest dissappoinment in my life coming o...
3,Negative,For the biggest male dissappoinment in my life...
4,Negative,the biggest dissappoinment in my life came bac...
...,...,...
7995,Irrelevant,Latest news about Xbox Geek! paper.li / XboxGe...
7996,Irrelevant,Latest xbox geek! paper.li / XboxGeekNews / 1....
7997,Irrelevant,The latest xbox hack blog! paper.li/XboxGeekNe...
7998,Irrelevant,The latest xbox from geek news! smart paper. l...


In [16]:
nlp = spacy.load("en_core_web_md")


In [17]:
def lemmatization(text):
    doc = nlp(text)
    lemmaList = [word.lemma_ for word in doc]
    return ' '.join(lemmaList)

df["lemma"] = df["tweet"].apply(lemmatization)
df

,sentiment,tweet,lemma
0,Negative,the biggest dissappoinment in my life came out...,the big dissappoinment in my life come out a y...
1,Negative,The biggest disappointment of my life came a y...,the big disappointment of my life come a year ...
2,Negative,the biggest dissappoinment in my life coming o...,the big dissappoinment in my life come out a y...
3,Negative,For the biggest male dissappoinment in my life...,for the big male dissappoinment in my life com...
4,Negative,the biggest dissappoinment in my life came bac...,the big dissappoinment in my life come back la...
...,...,...,...
7995,Irrelevant,Latest news about Xbox Geek! paper.li / XboxGe...,late news about Xbox Geek ! paper.li / XboxGee...
7996,Irrelevant,Latest xbox geek! paper.li / XboxGeekNews / 1....,late xbox geek ! paper.li / XboxGeekNews / 1 ....
7997,Irrelevant,The latest xbox hack blog! paper.li/XboxGeekNe...,the late xbox hack blog ! paper.li/XboxGeekNew...
7998,Irrelevant,The latest xbox from geek news! smart paper. l...,the late xbox from geek news ! smart paper . l...


In [ ]:
def remove_stopwords(text):
    doc = nlp(text)
    no_stopwords = [word.text for word in doc if not word.is_stop and not word.is_punct]
    return ' '.join(no_stopwords)

df['final_tweet'] = df["lemma"].apply(remove_stopwords)
df.drop(columns=["lemma","tweet"],inplace=True)
df

In [ ]:
df = pd.concat([df_game,df],axis=1)
df

,Amazon,Borderlands,CallOfDutyBlackopsColdWar,Overwatch,Xbox(Xseries),Amazon,Borderlands,CallOfDutyBlackopsColdWar,Overwatch,Xbox(Xseries),sentiment,final_tweet
23,0,1,0,0,0,0,1,0,0,0,Negative,big dissappoinment life come year ago fuck bor...
24,0,1,0,0,0,0,1,0,0,0,Negative,big disappointment life come year ago
26,0,1,0,0,0,0,1,0,0,0,Negative,big dissappoinment life come year ago fuck bor...
27,0,1,0,0,0,0,1,0,0,0,Negative,big male dissappoinment life come hang year ti...
28,0,1,0,0,0,0,1,0,0,0,Negative,big dissappoinment life come year ago fuck bor...
...,...,...,...,...,...,...,...,...,...,...,...,...
10788,0,0,0,0,1,0,0,0,0,1,Irrelevant,late news Xbox Geek paper.li XboxGeekNews 1 th...
10789,0,0,0,0,1,0,0,0,0,1,Irrelevant,late xbox geek paper.li XboxGeekNews 1 thank c...
10790,0,0,0,0,1,0,0,0,0,1,Irrelevant,late xbox hack blog paper.li/XboxGeekNews/1 th...
10791,0,0,0,0,1,0,0,0,0,1,Irrelevant,late xbox geek news smart paper li xboxgeeknew...


In [ ]:
x = df.drop(columns=["sentiment"])
y = df["sentiment"]

y.shape,x.shape

((8000,), (8000, 11))

In [ ]:
#
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(x["final_tweet"]).toarray()

In [ ]:
vectorized_df = pd.DataFrame(tfidf_matrix,columns=tfidf.get_feature_names_out())
vectorized_df

,00,000,01,02,03,05,06,10,100,1000,...,zxxxvid,zxxxvids,zyfapoihpy,ееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееееее,июля,сетью,третьарце,اللعبه,حبيت,خلاص
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
x.drop(columns=["final_tweet"], inplace=True)


In [ ]:
x

,Amazon,Borderlands,CallOfDutyBlackopsColdWar,Overwatch,Xbox(Xseries),Amazon,Borderlands,CallOfDutyBlackopsColdWar,Overwatch,Xbox(Xseries)
23,0,1,0,0,0,0,1,0,0,0
24,0,1,0,0,0,0,1,0,0,0
26,0,1,0,0,0,0,1,0,0,0
27,0,1,0,0,0,0,1,0,0,0
28,0,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...
10788,0,0,0,0,1,0,0,0,0,1
10789,0,0,0,0,1,0,0,0,0,1
10790,0,0,0,0,1,0,0,0,0,1
10791,0,0,0,0,1,0,0,0,0,1


In [ ]:
x= pd.concat([x,vectorized_df])
x

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
x.fillna(0)

In [ ]:
x_train,y_train,x_text,y_text = train_test_split(x,y,test_size=0.2,random_state=42)